# 02 · 单次完整回测

In [ ]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
ROOT = next((candidate for candidate in (start, *start.parents) if (candidate / 'vbt').exists()), start)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'项目根目录: {ROOT}')

## 1. 配置与数据（默认十年生产对齐口径）

In [ ]:
from vbt.config import load_backtest_config, load_strategy_config
from vbt.adapters import VBTDataLoader
config = load_backtest_config()
params = load_strategy_config()
data = VBTDataLoader(start_date=config['start_date'], end_date=config['end_date']).load_baseline_aligned(
    config['baseline_path'], initial_capital=config['initial_capital'])
config

## 2. 回测与绩效

In [ ]:
from vbt.engine import VBTEngine, PerformanceCalculator, ReportGenerator
from vbt.strategies import DividendLowVolStrategy
engine = VBTEngine(data=data, strategy=DividendLowVolStrategy(params), initial_capital=config['initial_capital'],
    commission=config['commission'], min_commission=config['min_commission'], stamp_duty_before=config['stamp_duty_before_2023_08_28'],
    stamp_duty_after=config['stamp_duty_after_2023_08_28'], slippage=config['slippage'], backtest_config=config)
results = engine.run()
perf = PerformanceCalculator(results)
perf.compute_metrics()

## 3. 净值、回撤与持仓热力图

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
nav = results.nav
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
nav.plot(ax=axes[0], title='组合资产')
(nav / nav.cummax() - 1).plot(ax=axes[1], title='回撤', color='firebrick')
plt.tight_layout()
active = results.positions.loc[:, results.positions.max().gt(0)].resample('ME').last()
plt.figure(figsize=(14, max(4, len(active.columns) * .22)))
sns.heatmap(active.T, cmap='YlGnBu', vmin=0)
plt.title('月末持仓权重热力图')

## 4. 导出 Markdown、HTML 与 Parquet

In [ ]:
ReportGenerator(results, perf, params).archive(config['output_dir'])